In [1]:
!pip install nltk scikit-learn pandas matplotlib

In [2]:
import pandas as pd
import numpy as np
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import pickle

In [3]:
from google.colab import files
uploaded = files.upload()

Saving True.csv to True.csv
Saving Fake.csv to Fake.csv


In [4]:
fake = pd.read_csv("Fake.csv")
real = pd.read_csv("True.csv")

fake["label"] = 0
real["label"] = 1
data = pd.concat([fake, real])
data = data[["text", "label"]].dropna()
data.reset_index(drop=True, inplace=True)
data.head()

,text,label
0,Donald Trump just couldn t wish all Americans ...,0
1,House Intelligence Committee Chairman Devin Nu...,0
2,"On Friday, it was revealed that former Milwauk...",0
3,"On Christmas day, Donald Trump announced that ...",0
4,Pope Francis used his annual Christmas Day mes...,0


In [5]:
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def clean_text(text):
    text = text.lower()
    text = "".join([char for char in text if char not in string.punctuation])
    words = text.split()
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    return " ".join(words)

data["clean_text"] = data["text"].apply(clean_text)
data.head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,text,label,clean_text
0,Donald Trump just couldn t wish all Americans ...,0,donald trump wish american happi new year leav...
1,House Intelligence Committee Chairman Devin Nu...,0,hous intellig committe chairman devin nune go ...
2,"On Friday, it was revealed that former Milwauk...",0,friday reveal former milwauke sheriff david cl...
3,"On Christmas day, Donald Trump announced that ...",0,christma day donald trump announc would back w...
4,Pope Francis used his annual Christmas Day mes...,0,pope franci use annual christma day messag reb...


In [6]:
X = data["clean_text"]
y = data["label"]
tfidf = TfidfVectorizer(max_features=5000)
X_vec = tfidf.fit_transform(X).toarray()
X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42)

In [7]:
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 0.9883073496659243
F1 Score: 0.9876659227064489


In [8]:
with open("news_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

In [11]:
file_path = input("Enter path to model file (e.g., /content/news_model.pkl): ")
with open(file_path, "rb") as f:
    loaded_model = pickle.load(f)
print("Model loaded successfully!")

Enter path to model file (e.g., /content/news_model.pkl): /content/news_model.pkl
Model loaded successfully!


In [13]:
!pip install streamlit


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 4.7 MB/s eta 0:00:00


In [14]:
# 💡 Streamlit UI Code
# You can copy this into a file named `app.py` and run it using: streamlit run app.py

import streamlit as st
import pickle
import string
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import nltk

nltk.download('stopwords')

# Load model and vectorizer
with open("news_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("tfidf_vectorizer.pkl", "rb") as f:
    tfidf = pickle.load(f)

def clean_input(text):
    stemmer = PorterStemmer()
    stop_words = set(stopwords.words('english'))

    text = text.lower()
    text = "".join([char for char in text if char not in string.punctuation])
    words = text.split()
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    return " ".join(words)

st.title("📰 Fake News Detector")

user_input = st.text_area("Enter a news article:")

if st.button("Predict"):
    cleaned = clean_input(user_input)
    vectorized = tfidf.transform([cleaned])
    prediction = model.predict(vectorized)
    st.write("### 🔍 This article is:", "🟢 Real" if prediction[0] == 1 else "🔴 Fake")


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
2025-06-19 06:55:25.932 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-19 06:55:26.216 
  command:

    streamlit run /usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-06-19 06:55:26.217 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-19 06:55:26.220 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-19 06:55:26.221 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-19 06:55:26.225 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-19 06:55:26.228 Thread 'MainThread': missing S

In [15]:
from google.colab import files
files.download('news_model.pkl')
files.download('tfidf_vectorizer.pkl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
pip install streamlit nltk


In [17]:
streamlit_code = '''\
import streamlit as st
import pickle
import string
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import nltk

nltk.download('stopwords')

# Load model and vectorizer
with open("news_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("tfidf_vectorizer.pkl", "rb") as f:
    tfidf = pickle.load(f)

def clean_input(text):
    stemmer = PorterStemmer()
    stop_words = set(stopwords.words('english'))

    text = text.lower()
    text = "".join([char for char in text if char not in string.punctuation])
    words = text.split()
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    return " ".join(words)

st.title("📰 Fake News Detector")

user_input = st.text_area("Enter a news article:")

if st.button("Predict"):
    cleaned = clean_input(user_input)
    vectorized = tfidf.transform([cleaned])
    prediction = model.predict(vectorized)
    st.write("### 🔍 This article is:", "🟢 Real" if prediction[0] == 1 else "🔴 Fake")
'''

# Save the code into a Python file
with open("app.py", "w") as f:
    f.write(streamlit_code)


In [18]:
from google.colab import files
files.download("app.py")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>